Running on Colab, will download and install neccesary packages.
upload the "newBenchmarks_dataset.xlsx" in the main folder

In [ ]:
!pip install tensorboardX
!pip install pysr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 22.8 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/AndreasMadsen/stable-nalu.git

Cloning into 'stable-nalu'...
remote: Enumerating objects: 3003, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 3003 (delta 57), reused 109 (delta 55), pack-reused 2892 (from 1)
Receiving objects: 100% (3003/3003), 14.82 MiB | 15.28 MiB/s, done.
Resolving deltas: 100% (2271/2271), done.


In [ ]:
cd stable-nalu

/content/stable-nalu


In [ ]:
!python setup.py develop

/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'test_suite'
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'tests_require'
  warnings.warn(msg)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        ********************************************************************************

!!
  easy_install.initialize_options(self)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDepre

In [ ]:
import numpy as np
import pandas as pd
import pickle
from datetime import datetime

In [ ]:
# import pysr

In [ ]:
import torch
import torch.nn as nn
from stable_nalu.layer.re_regualized_linear_nac import ReRegualizedLinearNACLayer
from stable_nalu.layer.re_regualized_linear_nalu import ReRegualizedLinearNALULayer

/content/stable-nalu/stable_nalu/functional/gumbel.py:26: SyntaxWarning: invalid escape sequence '\i'
  tau: the temperature used, must be tau \in (0, \infty]. tau < 1
/content/stable-nalu/stable_nalu/layer/nac.py:13: SyntaxWarning: invalid escape sequence '\h'
  Asumming \hat{w} and \hat{m} are sampled from a uniform


In [ ]:
SEED=1#4
rnd=np.random.default_rng(1)
torch.manual_seed(SEED)

In [ ]:
!ls ../

sample_data  stable-nalu


load the data for training. the data is collected by running the original paper.

In [ ]:
data= pd.read_excel(r"../newBenchmarks_dataset.xlsx","ideal_gas")
data['C'] = 1
data.shape

(30, 12)

In [ ]:
data

,Unnamed: 0,n,T,V,y,Unnamed: 5,Unnamed: 6,t(without noise),Unnamed: 8,Unnamed: 9,Unnamed: 10,C
0,0,26.860299,333.308025,53.255204,1298.568081,NaN,0,1397.669759,NaN,1397.669759,3.410605e-12,1
1,1,54.780396,361.112348,88.436616,1966.328795,NaN,1,1859.710791,NaN,1859.710791,2.046363e-12,1
2,2,18.091794,369.319481,3.909246,12332.772583,NaN,2,14210.222317,NaN,14210.222317,8.185452e-11,1
3,3,20.409633,369.127887,49.711568,1113.821284,NaN,3,1259.982789,NaN,1259.982789,9.094947e-13,1
4,4,42.240452,390.692602,20.320927,6040.255777,NaN,4,6751.966151,NaN,6751.966151,5.456968e-12,1
5,5,1.532568,324.938408,94.239958,42.659309,NaN,5,43.933491,NaN,43.933491,1.776357e-13,1
6,6,37.701015,384.429803,67.376748,1668.973970,NaN,6,1788.422266,NaN,1788.422266,2.273737e-13,1
7,7,56.285506,390.620308,39.080054,5440.829281,NaN,7,4677.418883,NaN,4677.418883,9.094947e-13,1
8,8,8.482414,324.040557,97.520973,251.914205,NaN,8,234.331589,NaN,234.331589,2.273737e-13,1
9,9,92.571299,317.609510,3.776232,53868.575239,NaN,9,64732.332409,NaN,64732.332409,1.382432e-10,1


train, validation split indexes

In [ ]:
unique_train_indexes = list(rnd.choice(data.shape[0], int(data.shape[0]*0.75), replace=False,))
unique_train_indexes

[np.int64(22),
 np.int64(20),
 np.int64(8),
 np.int64(4),
 np.int64(29),
 np.int64(14),
 np.int64(18),
 np.int64(15),
 np.int64(17),
 np.int64(11),
 np.int64(27),
 np.int64(25),
 np.int64(2),
 np.int64(9),
 np.int64(0),
 np.int64(5),
 np.int64(21),
 np.int64(16),
 np.int64(12),
 np.int64(24),
 np.int64(19),
 np.int64(26)]

In [ ]:
unique_test_indexes = [_ for _ in range(data.shape[0]) if _ not in unique_train_indexes]
len(unique_train_indexes),len(unique_test_indexes)

(22, 8)

In [ ]:
idealGas_train_X = data.loc[unique_train_indexes,['n','T','V','C']].values
idealGas_train_y = data.loc[unique_train_indexes,['y']].values#'Obs''t(without noise)'

idealGas_test_X = torch.from_numpy(data.loc[unique_test_indexes,['n','T','V','C']].values)
idealGas_test_y = torch.from_numpy(data.loc[unique_test_indexes,['y']].values)


In [ ]:
idealGas_train_X.shape,idealGas_train_y.shape,idealGas_test_X.shape,idealGas_test_y.shape

((22, 4), (22, 1), torch.Size([8, 4]), torch.Size([8, 1]))

In [ ]:
pd.DataFrame(idealGas_test_X)

,0,1,2,3
0,54.780396,361.112348,88.436616,1.0
1,20.409633,369.127887,49.711568,1.0
2,37.701015,384.429803,67.376748,1.0
3,56.285506,390.620308,39.080054,1.0
4,85.028936,366.607447,95.698817,1.0
5,19.759484,344.550271,77.735539,1.0
6,78.267053,336.696166,4.357185,1.0
7,43.095718,342.170389,62.111742,1.0


In [ ]:
from torch.utils.data import Dataset, DataLoader
loader = DataLoader(list(zip(idealGas_train_X,idealGas_train_y)), shuffle=True, batch_size=5)

In [ ]:
kwargs_nalu={'eps': 1e-07,'nac_oob': 'clip','regualizer_shape': 'linear','regualizer_z': 0,'mnac_epsilon': 0,'nalu_bias': True,'nalu_two_nac': True,'nalu_two_gate': False,'nalu_mul': 'mnac','nalu_gate': 'normal'}


In [ ]:
class MyModel(nn.Module):
    def __init__(self, in_feats,h_feats, out_feat):
        super(MyModel, self).__init__()
        self.net0=nn.Linear(in_feats , h_feats,bias=False)
        self.net1=nn.Linear(h_feats , h_feats,bias=False)
        # # self.net1_1=nn.Linear(h_feats*2 , h_feats*2,bias=False)
        self.leaKyrelu=torch.nn.LeakyReLU(0.1)
        # self.net_nac=ReRegualizedLinearNACLayer(in_features=h_feats,out_features=h_feats,**kwargs_nac)
        self.net_nalu=ReRegualizedLinearNALULayer(in_features=h_feats,out_features=out_feat,**kwargs_nalu)
    def forward(self,inputs_):
        # h=inputs_.float()
        h=self.net0(inputs_.float())
        h=self.leaKyrelu(h)
        # # h=self.net1_1(h)
        # # h=self.leaKyrelu(h)
        h=self.net1(h)
        h=self.leaKyrelu(h)
        # # h=self.net_nac(h)
        h=self.net_nalu(h)
        return h.double()
    def reset_parameters(self):
        torch.nn.init.xavier_uniform(self.net0.weight)
        # torch.nn.init.xavier_uniform(self.net1.weight)
        # # torch.nn.init.xavier_uniform(self.net1_1.weight)
        self.net_nalu.reset_parameters()
        # self.net_nac.reset_parameters()
    def regualizer(self):
        self.net_nalu.regualizer()
        # self.net_nac.regualizer()

In [ ]:
net=MyModel(4,4,1)
net.reset_parameters()

/tmp/ipykernel_3134/3205468057.py:22: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(self.net0.weight)


In [ ]:
print(f"start:{datetime.now()}")

start:2026-06-28 00:57:45.762972


In [ ]:
# first run with lr=0.01 for 200000, then decrease to 0.001
optim = torch.optim.RMSprop(net.parameters(), lr=0.001, weight_decay=1e-5)#

In [ ]:
best_loss=np.inf
criterion = torch.nn.L1Loss()
for i in range(200000):
    loss_list=[]
    # net.train()
    for X_t,y_t in loader:
        optim.zero_grad()
        out=net(X_t)
        loss_train_criterion = criterion(y_t,out)
#         loss_train_regualizer =10 * r_w_scale * regualizers['W'] + regualizers['g'] + 0 * regualizers['z'] + 1 * regualizers['W-OOB']
        loss = loss_train_criterion #+ loss_train_regualizer
        loss.retain_grad()
        loss_list.append(loss.item())
        #################
        mea = torch.mean(torch.abs(y_t - out))
        loss.backward()
        optim.step()
    if i%1==0:
        with torch.no_grad():
            test_loss=criterion(idealGas_test_y,net(idealGas_test_X)).item()
            if test_loss<best_loss:
                best_loss=test_loss
                torch.save(net,f'../models/model_idealGas_4inputs_{i}.pt')

                print(f'saved at {i} with error {np.array(loss_list).mean()}, {best_loss}')

saved at 802 with error 94.4468179466972, 212.68554622346736
saved at 1091 with error 84.2585720929738, 212.34084254915095
saved at 1242 with error 91.60442361516557, 209.84301692659236
saved at 1359 with error 90.01440158038491, 208.67129451448298
saved at 1508 with error 82.12826949037925, 207.3878899134273
saved at 1591 with error 83.52314227210326, 205.78191282284686
saved at 1766 with error 86.53234978766503, 205.1662106119014


In [ ]:
print(f"end:{datetime.now()}")

end:2026-06-28 01:07:11.649425


In [ ]:
# runtime on colab cpu: first 200000 iter was 10min+10min

In [ ]:
# saved at 0 with error 5790.363335905874, 7247.552546189302
# saved at 1 with error 5763.0294846117895, 7241.816197675788
# saved at 2 with error 5757.135778769619, 7240.301098962217
# saved at 3 with error 5755.393255583331, 7239.421918009543
# saved at 4 with error 5754.328108623393, 7234.4430374141275
# saved at 5 with error 5745.639646680287, 7004.485935554975
# saved at 8 with error 5556.6950240635915, 6372.147472432959
# saved at 10 with error 5472.095742058976, 5581.586132100928
# saved at 16 with error 4211.60091656355, 5575.411357931006
# saved at 17 with error 3689.3080016080016, 5530.70231252085
# saved at 18 with error 3635.3133475098116, 5383.787670186865
# saved at 19 with error 3589.7145623324336, 5367.4180717981935
# saved at 20 with error 3529.5556739490885, 5357.035945943701
# saved at 21 with error 3467.0097308993704, 5244.782512716162
# saved at 23 with error 3334.5180732954996, 5149.267543844092
# saved at 24 with error 3253.7640463623497, 5070.768353403181
# saved at 25 with error 3189.505094755147, 5016.238121687993
# saved at 28 with error 4165.212046629954, 4899.91387647447
# saved at 29 with error 3074.698148463391, 4872.765169982707
# saved at 42 with error 3551.4582443247787, 4755.19805616197
# saved at 44 with error 3509.0583479631678, 4713.111355722517
# saved at 52 with error 3713.8928080835535, 4415.639311984014
# saved at 56 with error 3041.617070598406, 4381.76031418128
# saved at 60 with error 3017.3069391786175, 4342.512847140264
# saved at 62 with error 3013.6899924380336, 4310.858802035284
# saved at 64 with error 3014.080624807335, 4278.970435336065
# saved at 76 with error 3181.8371003118245, 4246.846372914852
# saved at 78 with error 2953.825822623663, 4214.159109426082
# saved at 80 with error 2935.909611495428, 4178.975447011531
# saved at 82 with error 2906.5162981742283, 4139.874086690486
# saved at 86 with error 3174.7003668502457, 4123.551142049373
# saved at 88 with error 2865.0806475874724, 4109.554293826055
# saved at 90 with error 3137.435212965358, 4101.280390096248
# saved at 94 with error 2869.1343946345496, 4065.7167533164634
# saved at 98 with error 2829.8146092997026, 4036.843302083553
# saved at 116 with error 2684.5561721435365, 3973.0140462177064
# saved at 118 with error 3180.6699049753715, 3968.515430952814
# saved at 120 with error 3142.095310932853, 3962.3064236896303
# saved at 122 with error 2925.123167443786, 3852.578705336359
# saved at 134 with error 2836.271653516153, 3784.835857893732
# saved at 146 with error 2817.7959310515043, 3725.7747769611146
# saved at 154 with error 2823.4527754506967, 3695.45352223089
# saved at 160 with error 2811.2272333280293, 3662.045822645929
# saved at 164 with error 2812.8747238402702, 3625.030853773859
# saved at 166 with error 2767.310083296593, 3514.0285687701967
# saved at 170 with error 2739.0855990588357, 3439.9606099384096
# saved at 174 with error 2704.3099510067927, 3347.9967637317445
# saved at 178 with error 2664.416009783172, 3238.0382676379945
# saved at 182 with error 2617.1749892380853, 3216.934021498468
# saved at 184 with error 2601.505423621056, 2923.800783659723
# saved at 195 with error 2410.058299373809, 2453.2667145984437
# saved at 210 with error 1948.3245898618609, 1942.8863465953305
# saved at 217 with error 1818.6754269963626, 1707.1366541552975
# saved at 224 with error 1542.2280632721618, 1193.7568914751946
# saved at 263 with error 1461.805239281467, 1186.2993003297286
# saved at 1378 with error 725.6474161163006, 1164.0043832073584
# saved at 1525 with error 667.8998469454356, 1132.6093276202366
# saved at 1582 with error 693.2302220593399, 1120.3178139934912
# saved at 1648 with error 688.3581916848426, 1073.3650848852662
# saved at 1685 with error 704.2915574378139, 1027.6272079931764
# saved at 1807 with error 617.3832344369674, 941.0219613323582
# saved at 2016 with error 1236.7300088256102, 917.1478628552736
# saved at 2242 with error 1302.18190774723, 865.9203017413304
# saved at 2318 with error 1216.9000310965064, 788.4878905474827
# saved at 2505 with error 1236.310298492913, 671.467932051389
# saved at 2589 with error 1242.729669664344, 671.2721990979494
# saved at 2959 with error 1115.9357740447847, 544.9503997027562
# saved at 3669 with error 603.5715126352096, 504.82883209355487
# saved at 3777 with error 574.2374252637935, 491.25934875467976
# saved at 3968 with error 605.3974201420043, 474.2313259885665
# saved at 3990 with error 652.7900212689099, 461.27040374735554
# saved at 4036 with error 679.4948535193315, 420.28057456700606
# saved at 4101 with error 560.4542923342811, 404.99383819559773
# saved at 4107 with error 1217.892022255916, 265.6170384952287
# saved at 4548 with error 615.5868112353272, 242.39137853884088
# saved at 10836 with error 1171.1987020682607, 229.89790930055963
# saved at 17407 with error 468.0010395118659, 228.32173063290546
# saved at 35137 with error 540.0838048235008, 224.54327978395807
# saved at 36095 with error 491.3266639102972, 219.18237823613302
# saved at 51416 with error 472.2155980998257, 213.3791784680666



# saved at 802 with error 94.4468179466972, 212.68554622346736
# saved at 1091 with error 84.2585720929738, 212.34084254915095
# saved at 1242 with error 91.60442361516557, 209.84301692659236
# saved at 1359 with error 90.01440158038491, 208.67129451448298
# saved at 1508 with error 82.12826949037925, 207.3878899134273
# saved at 1591 with error 83.52314227210326, 205.78191282284686
# saved at 1766 with error 86.53234978766503, 205.1662106119014

In [ ]:
for name, param in net.named_parameters():
    if 'weight' in name:
        print(f"Layer: {name} | Size: {param.size()} | Values : {param.data}")


Layer: net0.weight | Size: torch.Size([4, 4]) | Values : tensor([[  3.4708,  -1.1012,   2.6776, -15.6628],
        [  3.0370,   0.2122, -12.5354,   1.8041],
        [  0.1279,  -0.6284,   4.6197,  23.6341],
        [  1.2386,   0.4179,  -1.5048,   6.7576]])
Layer: net1.weight | Size: torch.Size([4, 4]) | Values : tensor([[-0.1209, -0.1154, -0.0552,  0.0342],
        [-0.2217, -0.0136,  0.0909, -0.0286],
        [-0.3080,  0.8505,  0.5149,  0.7084],
        [-0.4922,  0.7513,  0.6074,  0.5766]])


In [ ]:
i

199999

In [ ]:
loss

tensor(65.7900, dtype=torch.float64, grad_fn=<MeanBackward0>)

load best model saved for measuring the performance on all datasets

In [ ]:
net1=torch.load('../models/model_idealGas_4inputs_1766.pt',weights_only=False)

In [ ]:
pd.DataFrame([net1(torch.from_numpy(idealGas_train_X)).detach().numpy().flatten(),idealGas_train_y.flatten()]).T

,0,1
0,964.080444,1000.014551
1,3861.563477,3767.343051
2,189.314178,251.914205
3,5943.354492,6040.255777
4,4025.290283,4120.114076
5,3827.086426,3719.237140
6,2111.975342,2160.518739
7,10345.527344,10491.175518
8,1477.369141,1361.824715
9,1261.402100,1210.726263


In [ ]:
pd.DataFrame([net1(idealGas_test_X).detach().numpy().flatten(),idealGas_test_y.detach().numpy().flatten()]).T

,0,1
0,1958.231689,1966.328795
1,1154.415527,1113.821284
2,1706.042725,1668.973970
3,4058.545654,5440.829281
4,2688.264648,2541.370419
5,717.279968,716.242136
6,42667.824219,42659.954129
7,1831.513672,1814.029868


generate 100 randome samples for extrapolation

In [ ]:
n_list=rnd.uniform(0,100,100)
T_list=rnd.uniform(300,400,100)
V_list=rnd.uniform(0,100,100)

In [ ]:
res_ = []
for n,T,V in zip(n_list,T_list,V_list):
    model = net1(torch.from_numpy(np.array([[n,T,V,1]]))).detach().numpy()[0][0]
    obs = n*8.314*T/V
    res_.append([n,T,V,model,obs])

In [ ]:
res_ = pd.DataFrame(res_,columns=['n','T','V','Model','Obs'])

In [ ]:
res_['diff'] = (res_['Obs'] - res_['Model']).abs()
res_['perc'] = res_['diff']/res_['Obs']
res_['diff'].mean(),res_['perc'].mean()

(np.float64(3014.031931464546), np.float64(0.2274118906142555))

In [ ]:
res_

,n,T,V,Model,Obs,diff,perc
0,48.519097,372.647361,15.435363,9088.225586,9738.765960,650.540374,0.066799
1,98.073720,336.500726,51.030325,4613.029297,5376.756153,763.726856,0.142042
2,96.165719,344.839631,14.400288,32156.474609,19145.923248,13010.551361,0.679547
3,72.478994,336.769957,71.737173,2918.953857,2828.858751,90.095106,0.031849
4,54.122686,310.973466,27.631301,3671.115479,5064.205874,1393.090396,0.275086
...,...,...,...,...,...,...,...
95,42.092139,318.679368,58.496625,1719.700928,1906.488365,186.787437,0.097975
96,66.498425,367.468940,6.529871,32485.500000,31112.678979,1372.821021,0.044124
97,45.592896,357.056460,5.217626,21936.472656,25940.071608,4003.598951,0.154340
98,58.651833,315.855503,21.139845,5106.282227,7285.817042,2179.534816,0.299148


run the model on train/val datasets

In [ ]:
ddff = pd.read_excel('../newBenchmarks_dataset.xlsx','ideal_gas')

In [ ]:
# ddff['V'] = 1/ddff['V']
ddff['C'] = 1

In [ ]:
res_=pd.DataFrame([net1(torch.from_numpy(ddff[['n','T','V','C']].values)).detach().numpy().flatten(),ddff['t(without noise)'].values]).T
res_.columns=['Model','t(without noise)']
res_['diff'] = (res_['t(without noise)'] - res_['Model']).abs()
res_['perc'] = res_['diff']/res_['t(without noise)']
res_['diff'].mean(),res_['perc'].mean()

(np.float64(864.1750601136413), np.float64(0.07248427974804586))